In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Lambda
from tensorflow.keras.preprocessing.text import Tokenizer

I0000 00:00:1794851937.177671    9858 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794851937.252492    9858 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794851939.430370    9858 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
text = "I love deep learning and machine learning"
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
sequence = tokenizer.texts_to_sequences([text])[0]
vocab_size = len(tokenizer.word_index) + 1

print("Word Index:", tokenizer.word_index)
print("Sequence:", sequence)
print("Vocabulary Size:", vocab_size)

Word Index: {'learning': 1, 'i': 2, 'love': 3, 'deep': 4, 'and': 5, 'machine': 6}
Sequence: [2, 3, 4, 1, 5, 6, 1]
Vocabulary Size: 7


In [3]:
X = []
y = []

for i in range(2, len(sequence) - 2):
    X.append([sequence[i-2], sequence[i-1], sequence[i+1], sequence[i+2]])
    y.append(sequence[i])

X = np.array(X)
y = np.array(y)

print("Training context:", X)
print("Target Words:", y)

Training context: [[2 3 1 5]
 [3 4 5 6]
 [4 1 6 1]]
Target Words: [4 1 5]


In [4]:
model = Sequential([
    Embedding(vocab_size, 10),
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),
    Dense(vocab_size, activation='softmax')
])

model.build(input_shape=(None, 4))
model.summary()

E0000 00:00:1794851940.715493    9858 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 4, 10)          │            70 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │            77 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 147 (588.00 B)

 Trainable params: 147 (588.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X, y, epochs=100, verbose=0)
print("Model training completed")

In [ ]:
predictions = model.predict(X, verbose=0)

for i in range(len(X)):
    predicted = np.argmax(predictions[i])
    print("Context:", [tokenizer.index_word[w] for w in X[i]])
    print("Actual word:", tokenizer.index_word[y[i]])
    print("Predicted word:", tokenizer.index_word.get(predicted, "Unknown"))
    print()